[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/06_residual_and_connections.ipynb)

# 06. Residual and connection structures — residual, LayerScale, HC, mHC

이전 mHC section은 doubly-stochastic matrix 하나만 만든 뒤 일반 stream mixing을 보여줘 실제 Hyper-Connections update에서 중요한 **H_pre, H_res, H_post와 branch F의 연결**이 빠져 있었다.

이번 버전은 plain residual에서 시작해 residual scale을 거쳐, multi-stream mHC의 `x_{l+1} = H_res x_l + H_post^T F(H_pre x_l)` 구조까지 실제 tensor 계산으로 연결한다.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


device: cuda


## 1. Plain residual connection

single-stream residual은 `y = x + F(x)`다. identity path가 branch를 우회해 signal/gradient가 직접 흐를 수 있게 한다.


In [2]:
x = torch.randn(2, 8, device=device)
branch = nn.Sequential(
    nn.Linear(8, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
).to(device)

residual_output = x + branch(x)
print("residual output:", residual_output.shape)


residual output: torch.Size([2, 8])


## 2. ReZero and LayerScale

ReZero는 residual branch에 learned scalar를 두고, LayerScale은 channel-wise learned scale을 둔다. 중요한 차이는 scale parameter가 training state라는 점이다.


In [3]:
rezero_alpha = nn.Parameter(
    torch.zeros(1, device=device)
)
layer_scale_gamma = nn.Parameter(
    1e-4 * torch.ones(8, device=device)
)

rezero_output = x + rezero_alpha * branch(x)
layerscale_output = x + layer_scale_gamma * branch(x)

print("ReZero initial change:", (rezero_output - x).abs().max().item())
print("LayerScale initial change:", (layerscale_output - x).abs().max().item())


ReZero initial change: 0.0
LayerScale initial change: 6.788969039916992e-05


## 3. Hyper-Connections expand one residual stream into multiple streams

residual state를 `n`개의 parallel streams로 두면 layer branch에 어떤 stream mixture를 입력할지 `H_pre`, 기존 streams를 다음 layer로 어떻게 섞을지 `H_res`, branch output을 각 stream에 얼마씩 쓸지 `H_post`로 표현할 수 있다.


In [4]:
batch_size = 2
num_streams = 3
channels = 8

streams = torch.randn(
    batch_size,
    num_streams,
    channels,
    device=device,
)

H_pre = torch.tensor(
    [0.2, 0.5, 0.8],
    device=device,
)
H_post = torch.tensor(
    [1.0, 0.5, 0.2],
    device=device,
)
H_res = torch.tensor(
    [
        [0.8, 0.1, 0.1],
        [0.2, 0.7, 0.1],
        [0.1, 0.2, 0.7],
    ],
    device=device,
)

branch_input = torch.einsum(
    "s,bsc->bc",
    H_pre,
    streams,
)
branch_output = branch(branch_input)

residual_mixed = torch.einsum(
    "ij,bjc->bic",
    H_res,
    streams,
)
branch_written = (
    H_post[None, :, None]
    * branch_output[:, None, :]
)

hyper_output = residual_mixed + branch_written

print("branch input:", branch_input.shape)
print("multi-stream output:", hyper_output.shape)


branch input: torch.Size([2, 8])
multi-stream output: torch.Size([2, 3, 8])


## 4. mHC constrains all three mappings

mHC에서는 mappings를 hidden residual state에서 dynamically 생성한 뒤 `H_pre = sigmoid(raw_pre)`, `H_post = 2 sigmoid(raw_post)`, `H_res = Sinkhorn-Knopp(raw_res)`로 제한한다. 특히 `H_res`는 non-negative이고 모든 row/column sum이 1인 doubly-stochastic matrix가 된다.


In [5]:
def sinkhorn(logits, iterations=20):
    matrix = logits.exp()

    for _ in range(iterations):
        matrix = matrix / matrix.sum(
            dim=-1,
            keepdim=True,
        )
        matrix = matrix / matrix.sum(
            dim=-2,
            keepdim=True,
        )

    return matrix


class TinyMHC(nn.Module):
    def __init__(self, num_streams=3, channels=8):
        super().__init__()

        self.num_streams = num_streams
        flattened_dim = num_streams * channels

        self.norm = nn.RMSNorm(flattened_dim)
        self.pre_projection = nn.Linear(
            flattened_dim,
            num_streams,
        )
        self.post_projection = nn.Linear(
            flattened_dim,
            num_streams,
        )
        self.res_projection = nn.Linear(
            flattened_dim,
            num_streams * num_streams,
        )

        self.branch = nn.Sequential(
            nn.Linear(channels, 2 * channels),
            nn.SiLU(),
            nn.Linear(2 * channels, channels),
        )

    def forward(self, streams):
        batch_size, num_streams, channels = streams.shape

        flattened = streams.reshape(batch_size, -1)
        normalized = self.norm(flattened)

        H_pre = torch.sigmoid(
            self.pre_projection(normalized)
        )
        H_post = 2 * torch.sigmoid(
            self.post_projection(normalized)
        )
        raw_res = self.res_projection(normalized).view(
            batch_size,
            num_streams,
            num_streams,
        )
        H_res = sinkhorn(raw_res)

        branch_input = torch.einsum(
            "bs,bsc->bc",
            H_pre,
            streams,
        )
        branch_output = self.branch(branch_input)

        residual_mixed = torch.einsum(
            "bij,bjc->bic",
            H_res,
            streams,
        )
        branch_written = (
            H_post[:, :, None]
            * branch_output[:, None, :]
        )

        output = residual_mixed + branch_written
        return output, H_pre, H_post, H_res


mhc = TinyMHC().to(device)
output, H_pre, H_post, H_res = mhc(streams)

print("H_pre:\n", H_pre)
print("H_post:\n", H_post)
print("H_res row sums:\n", H_res.sum(dim=-1))
print("H_res column sums:\n", H_res.sum(dim=-2))
print("mHC output:", output.shape)


H_pre:
 tensor([[0.4089, 0.4854, 0.3274],
        [0.4061, 0.5103, 0.5234]], device='cuda:0', grad_fn=<SigmoidBackward0>)
H_post:
 tensor([[1.6156, 0.9119, 1.4079],
        [1.3245, 1.2030, 1.0312]], device='cuda:0', grad_fn=<MulBackward0>)
H_res row sums:
 tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0', grad_fn=<SumBackward1>)
H_res column sums:
 tensor([[1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000]], device='cuda:0', grad_fn=<SumBackward1>)
mHC output: torch.Size([2, 3, 8])


## References and provenance

**Residual connection** — He et al. identity shortcut를 반영했다.

**ReZero / LayerScale** — learned residual scaling을 scalar/channel level로 구분했다.

**Hyper-Connections** — multi-stream residual state와 pre/residual/post mappings의 역할을 반영했다.

**mHC: Manifold-Constrained Hyper-Connections** — Xie et al. `x_{l+1}=H_res x_l + H_post^T F(H_pre x_l)`, RMS-normalized dynamic mappings, sigmoid-constrained H_pre, `2*sigmoid` H_post, Sinkhorn-Knopp doubly-stochastic H_res를 작은 PyTorch 형태로 반영했다. production mHC의 fused kernels와 initialization/scaling details는 생략했다.
